In [2]:
"""
Step 1: Combined multilingual parsing for IL-NER (Hindi + Urdu + Odia + Telugu).

IMPORTANT: Before running this, verify exact folder names by running:
    import os
    BASE_PATH = "/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets"
    for entry in sorted(os.listdir(BASE_PATH)):
        print(entry)

Adjust LANGUAGE_FOLDERS below to match exactly what you see (case-sensitive).
"""

import os
import json
from collections import Counter

BASE_PATH = "/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets"

# Adjust these to match your exact folder names (check casing/spelling first)
LANGUAGE_FOLDERS = {
    "Hindi":  "Hindi",
    "Urdu":   "Urdu",
    "Odia":   "Odia",     # might be "Oriya" - verify first
    "Telugu": "Telugu",
}


def parse_conll(file_path):
    sentences, tags = [], []
    tokens, tag_seq = [], []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                if tokens:
                    sentences.append(tokens)
                    tags.append(tag_seq)
                    tokens, tag_seq = [], []
            else:
                parts = line.split("\t")
                if len(parts) < 2:
                    continue
                token, tag = parts[0], parts[-1]
                tokens.append(token)
                tag_seq.append(tag)
    if tokens:
        sentences.append(tokens)
        tags.append(tag_seq)
    return sentences, tags


def load_all_languages(language_folders, base_path=BASE_PATH):
    """
    Returns per-language splits (for separate test evaluation) AND
    a combined training pool (for joint multilingual training).
    """
    per_language_data = {}

    for lang_name, folder_name in language_folders.items():
        train_path = os.path.join(base_path, folder_name, f"{folder_name}-train.txt")
        dev_path   = os.path.join(base_path, folder_name, f"{folder_name}-dev.txt")
        test_path  = os.path.join(base_path, folder_name, f"{folder_name}-test.txt")

        train_sents, train_tags = parse_conll(train_path)
        dev_sents, dev_tags     = parse_conll(dev_path)
        test_sents, test_tags   = parse_conll(test_path)

        per_language_data[lang_name] = {
            "train_sents": train_sents, "train_tags": train_tags,
            "dev_sents": dev_sents,     "dev_tags": dev_tags,
            "test_sents": test_sents,   "test_tags": test_tags,
        }

        print(f"{lang_name:>8} | train={len(train_sents):>6} | dev={len(dev_sents):>5} | test={len(test_sents):>5}")

    return per_language_data


def build_combined_pool(per_language_data):
    """
    Concatenates train and dev sentences across all languages into one pool.
    Test sets are kept SEPARATE per language for individual evaluation.
    """
    combined_train_sents, combined_train_tags = [], []
    combined_dev_sents, combined_dev_tags = [], []

    for lang_name, data in per_language_data.items():
        combined_train_sents.extend(data["train_sents"])
        combined_train_tags.extend(data["train_tags"])
        combined_dev_sents.extend(data["dev_sents"])
        combined_dev_tags.extend(data["dev_tags"])

    print(f"\nCombined training pool: {len(combined_train_sents)} sentences")
    print(f"Combined dev pool:      {len(combined_dev_sents)} sentences")

    return combined_train_sents, combined_train_tags, combined_dev_sents, combined_dev_tags


def build_shared_tag2id(combined_train_tags, save_dir="/kaggle/working"):
    unique_tags = sorted(set(t for seq in combined_train_tags for t in seq))
    tag2id = {tag: i for i, tag in enumerate(unique_tags)}
    id2tag = {i: tag for tag, i in tag2id.items()}

    save_path = os.path.join(save_dir, "tag2id_multilingual.json")
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(tag2id, f, ensure_ascii=False, indent=2)

    print(f"\nShared tag2id saved to: {save_path}")
    print(f"Num labels: {len(tag2id)}")
    print(tag2id)

    return tag2id, id2tag


# ---------------------------------------------------------------------
# Run everything
# ---------------------------------------------------------------------
per_language_data = load_all_languages(LANGUAGE_FOLDERS)

combined_train_sents, combined_train_tags, combined_dev_sents, combined_dev_tags = build_combined_pool(per_language_data)

tag2id, id2tag = build_shared_tag2id(combined_train_tags)
NUM_TAGS = len(tag2id)
O_TAG_INDEX = tag2id["O"]

print(f"\nReady for multilingual training.")
print(f"NUM_TAGS={NUM_TAGS}, O_TAG_INDEX={O_TAG_INDEX}")
print(f"\nPer-language test sets available in per_language_data[<lang>]['test_sents'/'test_tags']")
print(f"Languages: {list(per_language_data.keys())}")

   Hindi | train= 11076 | dev= 1389 | test= 1388
    Urdu | train=  8720 | dev= 1094 | test= 1096
    Odia | train= 12109 | dev= 1517 | test= 1519
  Telugu | train=  2993 | dev=  384 | test=  384

Combined training pool: 34898 sentences
Combined dev pool:      4384 sentences

Shared tag2id saved to: /kaggle/working/tag2id_multilingual.json
Num labels: 13
{'B-NEAR': 0, 'B-NEL': 1, 'B-NEN': 2, 'B-NEO': 3, 'B-NEP': 4, 'B-NETI': 5, 'I-NEAR': 6, 'I-NEL': 7, 'I-NEN': 8, 'I-NEO': 9, 'I-NEP': 10, 'I-NETI': 11, 'O': 12}

Ready for multilingual training.
NUM_TAGS=13, O_TAG_INDEX=12

Per-language test sets available in per_language_data[<lang>]['test_sents'/'test_tags']
Languages: ['Hindi', 'Urdu', 'Odia', 'Telugu']


In [4]:
!pip install -q --use-pep517 pytorch-crf seqeval transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
"""
Multilingual Tier 1 - Step A: Extract frozen MuRIL embeddings from the
COMBINED training/dev pool, and separately for each language's test set.

Requires from previous cell: combined_train_sents, combined_train_tags,
combined_dev_sents, combined_dev_tags, per_language_data, tag2id, id2tag,
NUM_TAGS, O_TAG_INDEX
"""

import os
import json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

MODEL_NAME = "google/muril-base-cased"
SAVE_PREFIX = "muril_multilingual"


@torch.no_grad()
def extract_embeddings_for_split(
    sentences,
    tag_lists,
    tokenizer,
    model,
    max_length=256,
    layer="last",
):
    model.eval()

    all_embeddings = []
    all_labels = []

    skipped_sentences = 0
    sentences_with_missing_word_ids = 0
    total_missing_word_ids = 0

    for sentence_index, (words, tags) in enumerate(zip(sentences, tag_lists)):
        encoding = tokenizer(
            words,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
        )

        word_ids = encoding.word_ids(batch_index=0)

        input_ids = encoding["input_ids"].to(DEVICE)
        attention_mask = encoding["attention_mask"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        if layer == "last":
            hidden = outputs.last_hidden_state[0]
        elif layer == "avg_last4":
            hidden = torch.stack(outputs.hidden_states[-4:]).mean(dim=0)[0]
        else:
            raise ValueError("layer must be 'last' or 'avg_last4'")

        hidden = hidden.cpu().numpy()

        # Gather every subword vector belonging to its original word ID.
        word_vecs = {}
        for subword_index, word_id in enumerate(word_ids):
            if word_id is None:
                continue
            word_vecs.setdefault(word_id, []).append(
                hidden[subword_index]
            )

        if not word_vecs:
            skipped_sentences += 1
            continue

        # Do NOT assume word IDs are contiguous.
        covered_word_ids = sorted(word_vecs.keys())

        expected_word_ids = set(range(len(words)))
        missing_word_ids = expected_word_ids - set(covered_word_ids)

        if missing_word_ids:
            sentences_with_missing_word_ids += 1
            total_missing_word_ids += len(missing_word_ids)

        sent_embeddings = []
        sent_labels = []

        for word_id in covered_word_ids:
            pooled_embedding = np.mean(
                word_vecs[word_id],
                axis=0,
            )

            sent_embeddings.append(pooled_embedding)
            sent_labels.append(tags[word_id])

        sent_embeddings = np.asarray(
            sent_embeddings,
            dtype=np.float32,
        )

        # Final safety check: one vector per retained label.
        assert sent_embeddings.shape[0] == len(sent_labels), (
            f"Sentence {sentence_index}: "
            f"{sent_embeddings.shape[0]} embeddings but "
            f"{len(sent_labels)} labels"
        )

        all_embeddings.append(sent_embeddings)
        all_labels.append(sent_labels)

    print(
        f"  Skipped empty sentences: {skipped_sentences}"
    )
    print(
        f"  Sentences with missing original words: "
        f"{sentences_with_missing_word_ids}"
    )
    print(
        f"  Total original words omitted: "
        f"{total_missing_word_ids}"
    )

    return all_embeddings, all_labels

def save_split(embs, labels, out_path):
    mismatches = sum(1 for e, l in zip(embs, labels) if e.shape[0] != len(l))
    assert mismatches == 0, f"{mismatches} alignment mismatches"

    np.savez_compressed(
        out_path,
        embeddings=np.array(embs, dtype=object),
        labels=np.array(labels, dtype=object),
    )
    total_tokens = sum(len(l) for l in labels)
    print(f"  Saved {len(embs)} sentences, {total_tokens} tokens -> {out_path}")


print(f"\n{'='*60}\nLoading {MODEL_NAME}\n{'='*60}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

# ---------------------------------------------------------------------
# Extract embeddings for COMBINED train and dev pools
# ---------------------------------------------------------------------
print("\nExtracting COMBINED train embeddings...")
train_embs, train_labels = extract_embeddings_for_split(
    combined_train_sents, combined_train_tags, tokenizer, model
)
save_split(train_embs, train_labels, f"{SAVE_PREFIX}_train.npz")

print("\nExtracting COMBINED dev embeddings...")
dev_embs, dev_labels = extract_embeddings_for_split(
    combined_dev_sents, combined_dev_tags, tokenizer, model
)
save_split(dev_embs, dev_labels, f"{SAVE_PREFIX}_dev.npz")

# ---------------------------------------------------------------------
# Extract embeddings SEPARATELY for each language's test set
# ---------------------------------------------------------------------
for lang_name, data in per_language_data.items():
    print(f"\nExtracting {lang_name} test embeddings...")
    test_embs, test_labels = extract_embeddings_for_split(
        data["test_sents"], data["test_tags"], tokenizer, model
    )
    save_split(test_embs, test_labels, f"{SAVE_PREFIX}_test_{lang_name.lower()}.npz")

del model, tokenizer
torch.cuda.empty_cache()

print("\nAll embeddings extracted and saved.")
print("Files created:")
for f in sorted(os.listdir(".")):
    if f.startswith(SAVE_PREFIX):
        print(f"  {f}")

Using device: cuda

Loading google/muril-base-cased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Extracting COMBINED train embeddings...
  Skipped empty sentences: 0
  Sentences with missing original words: 3
  Total original words omitted: 44
  Saved 34898 sentences, 676790 tokens -> muril_multilingual_train.npz

Extracting COMBINED dev embeddings...
  Skipped empty sentences: 0
  Sentences with missing original words: 0
  Total original words omitted: 0
  Saved 4384 sentences, 85460 tokens -> muril_multilingual_dev.npz

Extracting Hindi test embeddings...
  Skipped empty sentences: 0
  Sentences with missing original words: 0
  Total original words omitted: 0
  Saved 1388 sentences, 34404 tokens -> muril_multilingual_test_hindi.npz

Extracting Urdu test embeddings...
  Skipped empty sentences: 0
  Sentences with missing original words: 0
  Total original words omitted: 0
  Saved 1096 sentences, 24196 tokens -> muril_multilingual_test_urdu.npz

Extracting Odia test embeddings...
  Skipped empty sentences: 0
  Sentences with missing original words: 0
  Total original words omitte

In [8]:
"""
Multilingual Tier 1 - Step B: Train ONE BiLSTM-CRF on the combined
MuRIL embeddings, then evaluate separately on each language's test set.
"""

import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF
from seqeval.metrics import classification_report, f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_PREFIX = "muril_multilingual"

with open("tag2id_multilingual.json", "r", encoding="utf-8") as f:
    tag2id = json.load(f)
id2tag = {v: k for k, v in tag2id.items()}
NUM_TAGS = len(tag2id)


class EmbeddingNERDataset(Dataset):
    def __init__(self, npz_path, tag2id):
        data = np.load(npz_path, allow_pickle=True)
        self.embeddings = data["embeddings"]
        self.labels = data["labels"]
        self.tag2id = tag2id

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        emb = torch.tensor(self.embeddings[idx], dtype=torch.float32)
        label_ids = torch.tensor(
            [self.tag2id[t] for t in self.labels[idx]], dtype=torch.long
        )
        return emb, label_ids


def collate_fn(batch):
    embs, labels = zip(*batch)
    lengths = [e.shape[0] for e in embs]
    max_len = max(lengths)
    hidden_dim = embs[0].shape[1]

    padded_embs = torch.zeros(len(embs), max_len, hidden_dim)
    padded_labels = torch.zeros(len(embs), max_len, dtype=torch.long)
    mask = torch.zeros(len(embs), max_len, dtype=torch.bool)

    for i, (e, l) in enumerate(zip(embs, labels)):
        seq_len = e.shape[0]
        padded_embs[i, :seq_len] = e
        padded_labels[i, :seq_len] = l
        mask[i, :seq_len] = 1

    return padded_embs, padded_labels, mask


class BiLSTM_CRF(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_tags, num_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
            bidirectional=True, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, embeddings, labels=None, mask=None):
        lstm_out, _ = self.lstm(embeddings)
        lstm_out = self.dropout(lstm_out)
        emissions = self.fc(lstm_out)

        if labels is not None:
            loss = -self.crf(emissions, labels, mask=mask, reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for embs, labels, mask in loader:
        embs, labels, mask = embs.to(DEVICE), labels.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        loss = model(embs, labels=labels, mask=mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, id2tag):
    model.eval()
    all_preds, all_trues = [], []
    for embs, labels, mask in loader:
        embs, mask = embs.to(DEVICE), mask.to(DEVICE)
        pred_ids = model(embs, mask=mask)
        for i, seq_pred in enumerate(pred_ids):
            seq_len = mask[i].sum().item()
            true_ids = labels[i][:seq_len].tolist()
            all_preds.append([id2tag[p] for p in seq_pred])
            all_trues.append([id2tag[t] for t in true_ids])
    report = classification_report(all_trues, all_preds, digits=4, zero_division=0)
    f1 = f1_score(all_trues, all_preds, zero_division=0)
    return f1, report, all_preds, all_trues


# ---------------------------------------------------------------------
# Train ONE model on combined data
# ---------------------------------------------------------------------
print(f"\n{'='*60}\nTraining multilingual BiLSTM-CRF on combined MuRIL embeddings\n{'='*60}")

train_ds = EmbeddingNERDataset(f"{SAVE_PREFIX}_train.npz", tag2id)
dev_ds   = EmbeddingNERDataset(f"{SAVE_PREFIX}_dev.npz", tag2id)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)

input_dim = train_ds.embeddings[0].shape[1]
model = BiLSTM_CRF(input_dim, hidden_dim=256, num_tags=NUM_TAGS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_dev_f1 = 0.0
best_state = None
no_improve = 0
patience = 3
epochs = 20
start_time = time.time()

for epoch in range(1, epochs + 1):
    epoch_start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer)
    dev_f1, _, _, _ = evaluate(model, dev_loader, id2tag)
    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | dev_f1={dev_f1:.4f} | time={epoch_time:.1f}s")

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
torch.save(model.state_dict(), f"{SAVE_PREFIX}_bilstm_crf_best.pt")
total_train_time = time.time() - start_time

print(f"\nBest dev F1 (combined): {best_dev_f1:.4f}")
print(f"Total training time: {total_train_time:.1f}s\n")

# ---------------------------------------------------------------------
# Evaluate SEPARATELY on each language's test set
# ---------------------------------------------------------------------
per_language_results = {}

for lang_name in ["Hindi", "Urdu", "Odia", "Telugu"]:
    test_path = f"{SAVE_PREFIX}_test_{lang_name.lower()}.npz"
    test_ds = EmbeddingNERDataset(test_path, tag2id)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)

    test_f1, test_report, _, _ = evaluate(model, test_loader, id2tag)
    per_language_results[lang_name] = {
        "test_f1": round(test_f1, 4),
        "test_report": test_report,
        "num_test_sentences": len(test_ds),
    }

    print(f"\n{'='*60}\n{lang_name} Test Results\n{'='*60}")
    print(f"Test F1: {test_f1:.4f}")
    print(test_report)

# ---------------------------------------------------------------------
# Save combined summary
# ---------------------------------------------------------------------
summary = {
    "tier": "tier1_multilingual_combined",
    "backbone": "google/muril-base-cased (frozen)",
    "architecture": "BiLSTM-CRF",
    "best_dev_f1_combined": round(best_dev_f1, 4),
    "total_training_time_sec": round(total_train_time, 2),
    "per_language_test_results": per_language_results,
}

with open("tier1_multilingual_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"\n\n{'='*60}\nFINAL MULTILINGUAL TIER 1 SUMMARY\n{'='*60}")
print(f"{'Language':>10} | {'Test F1':>8} | {'Test Sentences':>14}")
for lang, res in per_language_results.items():
    print(f"{lang:>10} | {res['test_f1']:>8.4f} | {res['num_test_sentences']:>14}")

print("\nSaved to tier1_multilingual_results.json")


Training multilingual BiLSTM-CRF on combined MuRIL embeddings
Epoch  1 | train_loss=4.1028 | dev_f1=0.6207 | time=84.8s
Epoch  2 | train_loss=1.5081 | dev_f1=0.7235 | time=84.1s
Epoch  3 | train_loss=1.1384 | dev_f1=0.7701 | time=84.7s
Epoch  4 | train_loss=0.9584 | dev_f1=0.7808 | time=84.5s
Epoch  5 | train_loss=0.8503 | dev_f1=0.7784 | time=84.1s
Epoch  6 | train_loss=0.7782 | dev_f1=0.7930 | time=85.1s
Epoch  7 | train_loss=0.7234 | dev_f1=0.7842 | time=84.7s
Epoch  8 | train_loss=0.6795 | dev_f1=0.7940 | time=84.7s
Epoch  9 | train_loss=0.6343 | dev_f1=0.8136 | time=85.0s
Epoch 10 | train_loss=0.6078 | dev_f1=0.7929 | time=85.4s
Epoch 11 | train_loss=0.5591 | dev_f1=0.8087 | time=85.6s
Epoch 12 | train_loss=0.5389 | dev_f1=0.8019 | time=84.8s
Early stopping at epoch 12

Best dev F1 (combined): 0.8136
Total training time: 1017.5s


Hindi Test Results
Test F1: 0.8429
              precision    recall  f1-score   support

        NEAR     0.5397    0.5763    0.5574        59
       